# Fine-tuned baselines on Google Colab

Runs `analysis/finetune_baseline.py` (method: `docs/finetune-baseline.md`) on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Allow access to Google Drive when asked: each finished fold is saved to `MyDrive/model-swap-check-finetune/`.
3. After a disconnect, **Run all** again: finished folds are kept and training continues at the next one.
4. The last cell prints the results table and saves `results.zip` (predictions and run records, no message text) to the same Drive folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd /content
!test -d model-swap-check || git clone -q https://github.com/manjast/model-swap-check
%cd /content/model-swap-check
!git log -1 --format='%h %s'

In [ ]:
import subprocess
import sys
import tomllib

with open("pyproject.toml", "rb") as f:
    extra = tomllib.load(f)["project"]["optional-dependencies"]["finetune"]
# Colab's own PyTorch stays: it is built for Colab's GPU driver. The rest at the versions in pyproject.toml.
pins = [p for p in extra if not p.startswith("torch==")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", *pins], check=True)

In [ ]:
!swapcheck data
!mkdir -p /content/drive/MyDrive/model-swap-check-finetune
!test -L data/finetune || ln -s /content/drive/MyDrive/model-swap-check-finetune data/finetune

In [ ]:
!python analysis/finetune_baseline.py train modernbert 50

In [ ]:
!python analysis/finetune_baseline.py train modernbert 51

In [ ]:
!python analysis/finetune_baseline.py train setfit 50

In [ ]:
!python analysis/finetune_baseline.py train setfit 51

In [ ]:
!python analysis/finetune_baseline.py report
!zip -qr /content/drive/MyDrive/model-swap-check-finetune/results.zip baselines/modernbert baselines/setfit
!echo "results.zip saved to MyDrive/model-swap-check-finetune/"